# Pandas – Complete Reference Notebook

A structured walkthrough of Pandas covering Series, DataFrames, indexing, missing data, merging, grouping, string operations, and more.

**Table of Contents**
1. [Series](#1-series)
2. [DataFrame](#2-dataframe)
3. [Indexing & Selecting Data](#3-indexing--selecting-data)
4. [ufuncs & Index Alignment](#4-ufuncs--index-alignment)
5. [Handling Missing Values](#5-handling-missing-values)
6. [Hierarchical Indexing (MultiIndex)](#6-hierarchical-indexing-multiindex)
7. [Combining Datasets: concat](#7-combining-datasets-concat)
8. [Combining Datasets: merge & join](#8-combining-datasets-merge--join)
9. [Aggregation & GroupBy](#9-aggregation--groupby)
10. [Pivot Tables](#10-pivot-tables)
11. [Vectorized String Operations](#11-vectorized-string-operations)

## 1. Series

A `Series` is a 1-D labelled array. It has both a `.values` array and an `.index`.

In [ ]:
import pandas as pd
import numpy as np

# Default integer index
data = pd.Series([10, 20, 30, 40])
print(data)
print("values:", data.values)
print("index :", data.index)

In [ ]:
# Accessing elements
print("data[1]   :", data[1])     # single element
print("data[1:3]:\n", data[1:3]) # slice

### Custom Index

In [ ]:
data1 = pd.Series([10, 20, 30], index=['a', 'b', 'c'])
print(data1)
print("data1['b']:", data1['b'])

### Series from a Dictionary

In [ ]:
population = pd.Series({
    'California': 38332521,
    'Texas':      26448193,
    'Florida':    19552860
})
print(population)

### Series from a Scalar

In [ ]:
# The scalar is broadcast to every index position
print(pd.Series(5, index=[100, 200, 300]))

## 2. DataFrame

A `DataFrame` is a 2-D labelled table — think of it as a dict of same-length Series.

### From a Dictionary of Lists

In [ ]:
df = pd.DataFrame({
    'Name': ['Ram', 'Sam', 'Tom'],
    'Age':  [21, 22, 23]
})
print(df)

### From a Series

In [ ]:
df_pop = pd.DataFrame(population, columns=['Population'])
print(df_pop)

### From a List of Dictionaries

In [ ]:
# Keys become column names; missing keys produce NaN
data_list = [
    {'a': 1, 'b': 2},
    {'a': 3, 'b': 4}
]
print(pd.DataFrame(data_list))

# Mismatched keys → NaN for missing cells
print(pd.DataFrame([{'a': 1, 'b': 2}, {'b': 3, 'c': 4}]))

### From a NumPy Array

In [ ]:
arr = np.random.rand(3, 2)
print(pd.DataFrame(arr, columns=['A', 'B']))

### From a Structured NumPy Array

In [ ]:
A = np.zeros(3, dtype=[('A', 'i8'), ('B', 'f8')])
print(pd.DataFrame(A))

## 3. Indexing & Selecting Data

### Explicit vs Implicit Slicing on Series

When a Series has a custom index, slicing with `[]` can be ambiguous. Use `.loc` / `.iloc` to be explicit.

In [ ]:
data2 = pd.Series([0.25, 0.5, 0.75, 1.0], index=['a', 'b', 'c', 'd'])

print("Label slice data2['a':'c']:")
print(data2['a':'c'])        # explicit label slice (inclusive)

print("\nPosition slice data2[0:2]:")
print(data2[0:2])            # implicit integer slice (exclusive end)

### The Integer-Index Confusion

When an integer index exists, `[]` uses **labels**, but `[:]` slicing uses **positions** — a common gotcha.

In [ ]:
data3 = pd.Series(['A', 'B', 'C'], index=[1, 3, 5])
print("data3[1]   → label 1:", data3[1])   # label lookup
print("data3[1:3] → positions 1–2:\n", data3[1:3])  # positional slice

### `.loc` and `.iloc`

| Indexer | Meaning |
|---|---|
| `.loc` | Label-based (inclusive end) |
| `.iloc` | Integer position-based (exclusive end) |

In [ ]:
print(".loc[1:3]  (labels 1 and 3):")
print(data3.loc[1:3])

print("\n.iloc[1:3] (positions 1 and 2):")
print(data3.iloc[1:3])

### DataFrame Column Access & Dynamic Columns

In [ ]:
Area = pd.Series({
    'California': 423967, 'Texas': 695662,
    'New York':   141297, 'Florida': 170312,
    'Illinois':   149995
})
pop = pd.Series({
    'California': 38332521, 'Texas': 26448193,
    'New York':   19651127, 'Florida': 19552860,
    'Illinois':   12882135
})

df = pd.DataFrame({'Area': Area, 'pop': pop})

# Column access: attribute style or dict style
print("df.Area:")
print(df.Area)

# Add a computed column
df['density'] = df['pop'] / df['Area']
print("\nDataFrame with density column:")
print(df)

### `.values`, `.T` and Advanced Selection

In [ ]:
print("df.values (NumPy array):")
print(df.values)

print("\ndf.T (transposed):")
print(df.T)

In [ ]:
# Position-based selection with iloc
print("First 3 rows, first 2 cols (iloc):")
print(df.iloc[:3, :2])

# Label-based selection with loc
print("\nUp to Illinois, up to pop (loc):")
print(df.loc[:'Illinois', :'pop'])

# Filtering + column selection
print("\nHigh-density states:")
print(df.loc[df.density > 100, ['pop', 'density']])

### DataFrame Indexing Quick-Reference

| Syntax | What it selects |
|---|---|
| `df['col']` | A single column (Series) |
| `df['r1':'r2']` | Rows by label slice |
| `df[1:3]` | Rows by position slice |
| `df[bool_mask]` | Rows where mask is True |

In [ ]:
# Column
print("Column 'Area':")
print(df['Area'])

# Row label slice
print("\nRows Florida → Illinois:")
print(df['Florida':'Illinois'])

# Row position slice
print("\nRows at positions 1–2:")
print(df[1:3])

# Boolean filter
print("\nRows where density > 100:")
print(df[df.density > 100])

## 4. ufuncs & Index Alignment

Pandas passes ufuncs through to the underlying NumPy arrays, **preserving index labels**. When two Series are combined, Pandas first aligns them by index label — mismatches produce `NaN`.

In [ ]:
import numpy as np

ser = pd.Series([1, 4, 9], index=['A', 'B', 'C'])
print("np.sqrt preserves labels:")
print(np.sqrt(ser))

### Automatic Index Alignment

In [ ]:
A = pd.Series([2, 4, 6], index=[0, 1, 2])
B = pd.Series([1, 3, 5], index=[1, 2, 3])

print("A + B (NaN where indices don't overlap):")
print(A + B)

print("\nA.add(B, fill_value=0) — treat missing as 0:")
print(A.add(B, fill_value=0))

### DataFrame Alignment

In [ ]:
A = pd.DataFrame({'A': [1, 5], 'B': [11, 1]})
B = pd.DataFrame({'B': [4, 5, 9], 'A': [0, 8, 2], 'C': [9, 0, 6]})

print("A:")
print(A)
print("\nB:")
print(B)
print("\nA + B (aligned by label; NaN where shapes differ):")
print(A + B)

### DataFrame – Series Operations

In [ ]:
A_arr = np.array([[3, 8, 2, 4],
                  [2, 6, 4, 8],
                  [6, 1, 3, 8]])

df2 = pd.DataFrame(A_arr, columns=['Q', 'R', 'S', 'T'])

# Subtract first row from every row (row-wise broadcast)
print("Row-wise subtraction:")
print(df2 - df2.iloc[0])

# Subtract a column from every column (column-wise broadcast)
print("\nColumn-wise subtraction (axis=0):")
print(df2.subtract(df2['R'], axis=0))

### Pandas vs NumPy — Index Matching

In [ ]:
# NumPy matches by position
arr1 = np.array([10, 20, 30])
arr2 = np.array([1, 2, 3])
print("NumPy (position match):", arr1 + arr2)

# Pandas matches by label, not position
s1 = pd.Series([10, 20, 30], index=['A', 'B', 'C'])
s2 = pd.Series([1, 2, 3],   index=['C', 'A', 'B'])
print("\nPandas (label match):")
print(s1 + s2)

## 5. Handling Missing Values

Pandas represents missing data with `NaN` (float) or `None` (object). Pandas automatically converts `None` to `NaN` in numeric arrays.

### `None` vs `NaN`

In [ ]:
# None in a NumPy object array — arithmetic fails
arr_none = np.array([1, None, 3, 5])
print("dtype with None:", arr_none.dtype)  # object

# NaN in a float array — arithmetic propagates NaN
arr_nan = np.array([1, np.nan, 3, 4])
print("dtype with NaN :", arr_nan.dtype)   # float64
print("sum with NaN   :", arr_nan.sum())   # nan (propagates)
print("nansum         :", np.nansum(arr_nan))
print("nanmin         :", np.nanmin(arr_nan))
print("nanmax         :", np.nanmax(arr_nan))

### Pandas Converts `None` → `NaN`

In [ ]:
s = pd.Series([1, np.nan, 2, None])
print(s)  # both None and NaN become NaN

### Detecting Missing Values

In [ ]:
data = pd.Series([1, np.nan, 'hello', None])
print("isnull():")
print(data.isnull())

print("\nnotnull():")
print(data.notnull())

print("\nNon-null values only:")
print(data[data.notnull()])

### Removing Missing Values — `dropna()`

In [ ]:
df_na = pd.DataFrame([
    [1, np.nan, 2],
    [2, 3,      5],
    [np.nan, 4, 6]
])

print("Original:")
print(df_na)

print("\ndropna() — drop rows with any NaN:")
print(df_na.dropna())

print("\ndropna(axis=1) — drop columns with any NaN:")
print(df_na.dropna(axis=1))

In [ ]:
# how='all' — only drop if the entire row/col is NaN
df_na[3] = np.nan
print("Added all-NaN column 3:")
print(df_na)
print("\ndropna(axis=1, how='all'):")
print(df_na.dropna(axis=1, how='all'))

# thresh — keep rows with at least N non-null values
print("\ndropna(thresh=3) — keep rows with ≥3 non-null:")
print(df_na.dropna(thresh=3))

### Filling Missing Values — `fillna()`

In [ ]:
data = pd.Series([1, np.nan, 2, None, 3], index=list("abcde"))
print("Original:")
print(data)

print("\nfillna(0):")
print(data.fillna(0))

print("\nForward fill (ffill) — propagate last valid value:")
print(data.ffill())

print("\nBackward fill (bfill) — use next valid value:")
print(data.bfill())

## 6. Hierarchical Indexing (MultiIndex)

A `MultiIndex` lets a single Series or DataFrame carry data indexed by multiple keys simultaneously.

In [ ]:
import pandas as pd
import numpy as np

# Start with tuple-based index, then upgrade to a real MultiIndex
index = [
    ('California', 2000), ('California', 2010),
    ('Texas', 2000),      ('Texas', 2010)
]
population_data = [33871648, 37253956, 20851820, 25145561]

pop = pd.Series(population_data, index=index)
pop.index = pd.MultiIndex.from_tuples(pop.index)

# Name the levels
pop.index.names = ['State', 'Year']
print(pop)

### Accessing MultiIndex Data

In [ ]:
print("Single value (California, 2010):", pop['California', 2010])
print("\nAll years for California:")
print(pop['California'])
print("\nAll states for year 2010:")
print(pop[:, 2010])

### `unstack()` and `stack()`

In [ ]:
pop_df = pop.unstack()   # MultiIndex Series → 2-D DataFrame
print("unstack():")
print(pop_df)

print("\nstack() — back to Series:")
print(pop_df.stack())

### Creating MultiIndex — Three Methods

In [ ]:
# Method 1: from_arrays
mi1 = pd.MultiIndex.from_arrays([['a','a','b','b'], [1,2,1,2]])
print("from_arrays:", mi1)

# Method 2: from_tuples
mi2 = pd.MultiIndex.from_tuples([('a',1),('a',2),('b',1),('b',2)])
print("from_tuples:", mi2)

# Method 3: from_product (Cartesian product — most concise)
mi3 = pd.MultiIndex.from_product([['a','b'], [1,2]])
print("from_product:", mi3)

### MultiIndex DataFrame Example

In [ ]:
index   = pd.MultiIndex.from_product([[2023, 2024], [1, 2]], names=['Year','Visit'])
columns = pd.MultiIndex.from_product([['Bob','Alice'], ['HR','Temp']])

df_mi = pd.DataFrame(
    np.random.randint(30, 100, (4, 4)),
    index=index,
    columns=columns
)
print(df_mi)

print("\nBob's data:")
print(df_mi['Bob'])

print("\nBob's HR column:")
print(df_mi['Bob', 'HR'])

### Sorting, `reset_index()`, and `set_index()`

In [ ]:
# MultiIndex must be sorted for partial slicing to work
pop_sorted = pop.sort_index()
print("Sorted MultiIndex:")
print(pop_sorted)

print("\nreset_index() — flatten to regular columns:")
print(pop.reset_index(name='Population'))

## 7. Combining Datasets: `concat`

`pd.concat()` stacks objects along an axis. Default is axis=0 (row-wise).

In [ ]:
ser1 = pd.Series(['A', 'B', 'C'])
ser2 = pd.Series(['D', 'E', 'F'])

print("Series concat:")
print(pd.concat([ser1, ser2]))

In [ ]:
# Row-wise DataFrame concat (axis=0, default)
df1 = pd.DataFrame({'A': ['A1','A2'], 'B': ['B1','B2']})
df2 = pd.DataFrame({'A': ['A3','A4'], 'B': ['B3','B4']})

print("Row-wise concat:")
print(pd.concat([df1, df2]))

# Column-wise concat (axis=1)
df3 = pd.DataFrame({'A': ['A0','A1'], 'B': ['B0','B1']})
df4 = pd.DataFrame({'C': ['C0','C1'], 'D': ['D0','D1']})

print("\nColumn-wise concat (axis=1):")
print(pd.concat([df3, df4], axis=1))

### Handling Duplicate Indexes

In [ ]:
x = pd.DataFrame({'A': ['A0','A1']}, index=[0,1])
y = pd.DataFrame({'A': ['A2','A3']}, index=[0,1])

# ignore_index resets the index
print("ignore_index=True:")
print(pd.concat([x, y], ignore_index=True))

# keys add a MultiIndex to identify the source
print("\nkeys=['First','Second']:")
print(pd.concat([x, y], keys=['First','Second']))

In [ ]:
# Concatenating DataFrames with different columns
df5 = pd.DataFrame({'A':['A1','A2'], 'B':['B1','B2'], 'C':['C1','C2']})
df6 = pd.DataFrame({'B':['B3','B4'], 'C':['C3','C4'], 'D':['D3','D4']})

print("Outer join (default) — fills NaN for missing cols:")
print(pd.concat([df5, df6]))

print("\nInner join — keeps only shared columns:")
print(pd.concat([df5, df6], join='inner'))

## 8. Combining Datasets: `merge` & `join`

`pd.merge()` implements database-style joins. It automatically detects common column names.

### One-to-One Merge

In [ ]:
df1 = pd.DataFrame({
    'employee': ['Bob','Jake','Lisa','Sue'],
    'group':    ['Accounting','Engineering','Engineering','HR']
})
df2 = pd.DataFrame({
    'employee': ['Lisa','Bob','Jake','Sue'],
    'hire_date': [2004, 2008, 2012, 2014]
})

print("df1:"); print(df1)
print("\ndf2:"); print(df2)
print("\nMerged (auto-detects 'employee'):")
print(pd.merge(df1, df2))

### Many-to-One Merge

In [ ]:
df3 = pd.DataFrame({
    'employee': ['Bob','Jake','Lisa','Sue'],
    'group':    ['Accounting','Engineering','Engineering','HR']
})
df4 = pd.DataFrame({
    'group':      ['Accounting','Engineering','HR'],
    'supervisor': ['Carly','Guido','Steve']
})

print(pd.merge(df3, df4, on='group'))

### Many-to-Many Merge

In [ ]:
df5 = pd.DataFrame({
    'group':  ['Accounting','Accounting','Engineering','Engineering','HR','HR'],
    'skills': ['math','spreadsheets','coding','linux','spreadsheets','organization']
})

print(pd.merge(df3, df5, on='group'))

### `left_on` / `right_on` — Different Column Names

In [ ]:
df_salary = pd.DataFrame({
    'name':   ['Bob','Jake','Lisa','Sue'],
    'salary': [70000, 80000, 120000, 90000]
})

result = pd.merge(df3, df_salary, left_on='employee', right_on='name')
print("Before drop:")
print(result)

print("\nAfter dropping duplicate 'name' column:")
print(result.drop('name', axis=1))

### Merging on Index

In [ ]:
df1a = df1.set_index('employee')
df2a = df2.set_index('employee')

result_idx = pd.merge(df1a, df2a, left_index=True, right_index=True)
print("Merge on index:")
print(result_idx)

# .join() is a convenient shortcut for index-based merges
print("\ndf1a.join(df2a) — same result:")
print(df1a.join(df2a))

### Types of Joins

| `how=` | Keeps |
|---|---|
| `'inner'` (default) | Only rows with keys in **both** tables |
| `'outer'` | All rows from **either** table |
| `'left'` | All rows from the **left** table |
| `'right'` | All rows from the **right** table |

In [ ]:
df6 = pd.DataFrame({'name': ['Bob','Jake','Lisa'],  'rank': [1,2,3]})
df7 = pd.DataFrame({'name': ['Bob','Lisa','Sue'],   'rank': [3,1,4]})

for how in ['inner', 'outer', 'left', 'right']:
    print(f"how='{how}':")
    print(pd.merge(df6, df7, on='name', how=how, suffixes=(f'_{how[0]}L',f'_{how[0]}R')))
    print()

### Handling Overlapping Column Names

In [ ]:
df8 = pd.DataFrame({'name':['Bob','Jake','Lisa','Sue'], 'rank':[1,2,3,4]})
df9 = pd.DataFrame({'name':['Bob','Jake','Lisa','Sue'], 'rank':[3,1,4,2]})

# Default suffixes: _x and _y
print("Default suffixes:")
print(pd.merge(df8, df9, on='name'))

# Custom suffixes
print("\nCustom suffixes (_L, _R):")
print(pd.merge(df8, df9, on='name', suffixes=('_L','_R')))

## 9. Aggregation & GroupBy

### Series Aggregations

In [ ]:
marks = pd.Series([70, 80, 90, 60, 100])
print(f"sum  = {marks.sum()}")
print(f"mean = {marks.mean()}")
print(f"max  = {marks.max()}")
print(f"min  = {marks.min()}")

### DataFrame Aggregations

In [ ]:
df_agg = pd.DataFrame({'A': [10,20,30], 'B': [40,50,60]})

print("Column-wise mean:")
print(df_agg.mean())

print("\ndescribe() — full summary statistics:")
print(df_agg.describe())

### GroupBy

`groupby()` splits the data, applies a function, and combines the results.

In [ ]:
df_gb = pd.DataFrame({
    'Department': ['CSE','CSE','AI','AI','AI'],
    'Marks':      [80, 90, 70, 85, 95]
})
print("Data:")
print(df_gb)

print("\nGroup sum:")
print(df_gb.groupby('Department').sum())

print("\nGroup count:")
print(df_gb.groupby('Department').count())

print("\nMultiple aggregations:")
print(df_gb.groupby('Department')['Marks'].agg(['sum','mean','max','min']))

## 10. Pivot Tables

A pivot table is a 2-D grouped summary. It generalises GroupBy to two dimensions simultaneously.

In [ ]:
titanic = pd.DataFrame({
    'survived': [0,1,1,1,0,0,1,0],
    'sex':   ['male','female','female','female','male','male','female','male'],
    'class': ['Third','First','Third','First','Second','Third','Second','First'],
    'age':   [22,38,26,35,35,28,19,45],
    'fare':  [7.25,71.28,7.92,53.10,13.00,8.05,30.00,80.00]
})
print(titanic)

In [ ]:
# Default aggregation: mean
print("Mean survival rate by sex and class:")
print(titanic.pivot_table('survived', index='sex', columns='class'))

In [ ]:
# Sum
print("Total survivors by sex and class:")
print(titanic.pivot_table('survived', index='sex', columns='class', aggfunc='sum'))

# Multiple aggregations
print("\nMultiple aggregations (sum, mean, max):")
print(titanic.pivot_table('survived', index='sex', columns='class',
                          aggfunc=['sum','mean','max']))

In [ ]:
# margins=True — adds row/column totals
print("With margins (row/col totals):")
print(titanic.pivot_table('survived', index='sex', columns='class',
                          aggfunc='sum', margins=True))

## 11. Vectorized String Operations

Pandas exposes Python string methods element-wise via the `.str` accessor. Unlike list comprehensions, `.str` methods handle `None`/`NaN` gracefully.

In [ ]:
import pandas as pd

# List comprehension fails on None
data_list = ['peter', 'Paul', None, 'MARY']
# [s.capitalize() for s in data_list]  # → AttributeError

# .str accessor handles None automatically
names_raw = pd.Series(['peter', 'Paul', None, 'MARY', 'gUIDO'])
print(names_raw.str.capitalize())

### Common String Methods

In [ ]:
names = pd.Series([
    'Graham Chapman', 'John Cleese', 'Terry Gilliam',
    'Eric Idle', 'Terry Jones', 'Michael Palin'
])

print("lower():")
print(names.str.lower())

print("\nupper():")
print(names.str.upper())

print("\nlen():")
print(names.str.len())

print("\nstartswith('T'):")
print(names.str.startswith('T'))

print("\nendswith('n'):")
print(names.str.endswith('n'))

print("\nsplit():")
print(names.str.split())

### Pattern Matching & Replacement

In [ ]:
print("contains('Terry'):")
print(names.str.contains('Terry'))

print("\ncount('a') — occurrences of 'a':")
print(names.str.count('a'))

print("\nreplace('Terry', 'Mr.Terry'):")
print(names.str.replace('Terry', 'Mr.Terry'))

print("\nextract first word:")
print(names.str.extract('([A-Za-z]+)', expand=False))

### Vectorized Slicing & Splitting

In [ ]:
print("First 3 characters:")
print(names.str[0:3])

print("\nLast name (get last token from split):")
print(names.str.split().str.get(-1))

### `get_dummies()` — One-Hot Encode Delimited Strings

In [ ]:
df_dum = pd.DataFrame({'info': ['B|C|D', 'B|D', 'A|C', 'B|D']})
print("Original:")
print(df_dum)

print("\nget_dummies('|'):")
print(df_dum['info'].str.get_dummies('|'))